```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```

# RMI Sample Queries: BigQuery Admin (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/BigQuery_Admin_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FBigQuery_Admin_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/BigQuery_Admin_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/BigQuery_Admin_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **BigQuery Admin** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `boston_oct_2025_sample_data` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### bqa0_metadata_inventory.sql
**Business Question**: How can I quickly check the row count and storage size of all RMI tables using zero-cost metadata queries?
Product Stage: GA
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa0_metadata_inventory
/*
  This query utilizes INFORMATION_SCHEMA.PARTITIONS to provide a high-level 
  overview of table scale and data accumulation trends. 
  It processes 0 bytes because it scans system metadata rather than table data.
*/

SELECT
  table_name,
  CASE 
    WHEN partition_id IS NULL OR partition_id = '__UNPARTITIONED__' THEN 'UNPARTITIONED' 
    ELSE partition_id 
  END as partition_id,
  total_rows,
  ROUND(total_logical_bytes / POW(1024, 2), 2) as size_mb,
  last_modified_time
FROM `boston_oct_2025_sample_data.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name IN ('historical_travel_time', 'recent_roads_data', 'routes_status')
  AND partition_id != '__NULL__'
ORDER BY table_name, partition_id DESC;


### bqa1_scan_volume.sql
**Business Question**: Which users or service accounts are generating the highest scan volume against RMI tables this month?
Use Case: Enables cost governance by identifying 'heavy' consumers of the RMI dataset. Administrators can use this data to justify budget reallocations or suggest query optimizations to specific teams.
Product Stage: GA (Uses BigQuery INFORMATION_SCHEMA)
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa1_scan_volume
/*
  AUDIT PATTERN: Job Metadata Analysis
  This query scans the system-managed JOBS view. It calculates total data scanned 
  (billed bytes) for any query that mentions core RMI table names.
  
  Note: Replace 'region-us' with the specific region where your dataset resides.
*/

SELECT
  user_email,
  -- Convert bytes to GB for readable billing analysis
  SUM(total_bytes_billed) / POW(1024, 3) AS total_gb_billed,
  COUNT(*) AS job_count,
  -- Average scan size helps distinguish between 'many small queries' vs 'one massive scan'
  AVG(total_bytes_billed) / POW(1024, 3) AS avg_gb_per_job
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time BETWEEN TIMESTAMP_TRUNC(CURRENT_TIMESTAMP(), MONTH) AND CURRENT_TIMESTAMP()
  AND job_type = 'QUERY'
  -- Heuristic filter: Look for queries mentioning RMI core tables
  AND (
    query LIKE '%historical_travel_time%' 
    OR query LIKE '%recent_roads_data%' 
    OR query LIKE '%routes_status%'
  )
GROUP BY 1
ORDER BY total_gb_billed DESC
LIMIT 10;


### bqa2_cost_attribution.sql
**Business Question**: Identify any BigQuery jobs missing the mandatory 'rmisqlfactory_' prefix in their job IDs.
Use Case: Ensures compliance with project governance standards. Consistent job ID prefixing is required for accurate cost attribution and auditing of RMI-related analysis.
Product Stage: GA (Uses BigQuery INFORMATION_SCHEMA)
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa2_cost_attribution
/*
  NOTE: 'rmisqlfactory_' is the mandatory job ID prefix for this workspace.
  This allows administrators to filter billing logs and correlate spend 
  with specific personas or tools.
  
  SCOPE NOTE: Replace 'JOBS' with 'JOBS_BY_ORGANIZATION' if you have the 
  necessary permissions to audit spend across multiple projects.
*/

SELECT
  job_id,
  user_email,
  creation_time,
  total_bytes_billed,
  -- Provide the query text to help identify the source of the non-compliant job
  query
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY'
  -- Filter for jobs that missed the mandatory prefix
  AND NOT STARTS_WITH(job_id, 'rmisqlfactory_')
  -- Only audit jobs that were targeting RMI core tables
  AND (
    query LIKE '%historical_travel_time%' 
    OR query LIKE '%recent_roads_data%' 
    OR query LIKE '%routes_status%'
  )
ORDER BY creation_time DESC;


### bqa3_derived_resources.sql
**Business Question**: What tables or views in my project are derived from the core RMI dataset?
Use Case: Critical for lineage auditing and change management. Identifies 'shadow' analytical assets that may need to be updated or retired if the core RMI schema changes.
Product Stage: GA (Uses BigQuery INFORMATION_SCHEMA)
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa3_derived_resources
/*
  LINEAGE PATTERN: Metadata Dependency Mapping
  This query scans the project metadata to find any VIEW definition that 
  references RMI core tables, as well as any clones or snapshots 
  targeting 'rmi' or 'road' named resources.
*/

-- Replace `your-project.your-dataset` with the location of your analytical workspace.

SELECT
  table_schema AS dataset_id,
  table_name AS resource_name,
  'VIEW' AS type,
  -- view_definition allows the admin to see the exact transformation logic
  view_definition as lineage_detail
FROM `your-project.your-dataset.INFORMATION_SCHEMA.VIEWS`
WHERE (
    view_definition LIKE '%historical_travel_time%' 
    OR view_definition LIKE '%recent_roads_data%' 
    OR view_definition LIKE '%routes_status%'
  )

UNION ALL

-- Also identify Clones and Snapshots (cost-effective analytical patterns)
SELECT
  table_schema AS dataset_id,
  table_name AS resource_name,
  table_type AS type,
  'N/A (Check Table Metadata for Base Table Lineage)' AS lineage_detail
FROM `your-project.your-dataset.INFORMATION_SCHEMA.TABLES`
WHERE (table_name LIKE '%rmi%' OR table_name LIKE '%road%')
  AND table_type IN ('BASE TABLE', 'CLONE', 'SNAPSHOT')
  -- Exclude the raw source tables themselves
  AND table_name NOT IN ('historical_travel_time', 'recent_roads_data', 'routes_status')

ORDER BY dataset_id, resource_name;


### bqa4_query_patterns.sql
**Business Question**: What are the most frequent query patterns (joins, filters, JSON extractions) that could benefit from optimized downstream tables?
Use Case: Enables pro-active optimization. If many users are extracting the same JSON attribute or joining the same tables daily, the Admin can create a materialized view or flattened table to improve performance and reduce costs.
Product Stage: GA (Uses BigQuery INFORMATION_SCHEMA)
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa4_query_patterns
/*
  OPTIMIZATION PATTERN: Pattern Mining
  This query analyzes your recent job history to identify common access patterns.
  
  Note: Replace 'region-us' with your actual BigQuery region.
*/

WITH job_history AS (
  SELECT
    query,
    total_bytes_processed
  FROM `region-us`.INFORMATION_SCHEMA.JOBS
  WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
    AND job_type = 'QUERY'
    AND statement_type = 'SELECT'
    -- Heuristic: Focus on RMI-related queries
    AND (
      query LIKE '%historical_travel_time%' 
      OR query LIKE '%recent_roads_data%' 
      OR query LIKE '%routes_status%'
    )
),
patterns AS (
  SELECT
    query,
    -- Regex: Identify specific JSON attributes being extracted from 'route_attributes'
    REGEXP_EXTRACT_ALL(query, r"JSON_EXTRACT_SCALAR\(route_attributes, '([^']+)'\)") as extracted_attributes,
    -- Regex: Detect if the query performs SRI flattening (expensive unnest)
    REGEXP_CONTAINS(query, r"UNNEST\(speed_reading_intervals\)") as uses_sri_unnest,
    -- Regex: Detect common join patterns
    REGEXP_CONTAINS(query, r"JOIN\s+`[^`]+historical_travel_time`") AND REGEXP_CONTAINS(query, r"JOIN\s+`[^`]+routes_status`") as joins_hist_and_status
  FROM job_history
)
SELECT
  'Frequent Attribute Extraction' as pattern_type,
  attr as detail,
  COUNT(*) as frequency
FROM patterns, UNNEST(extracted_attributes) as attr
GROUP BY 1, 2

UNION ALL

SELECT
  'Heavy SRI Processing' as pattern_type,
  'Uses UNNEST(speed_reading_intervals)' as detail,
  COUNT(*) as frequency
FROM patterns
WHERE uses_sri_unnest
GROUP BY 1, 2

UNION ALL

SELECT
  'Common Table Joins' as pattern_type,
  'Joins historical_travel_time and routes_status' as detail,
  COUNT(*) as frequency
FROM patterns
WHERE joins_hist_and_status
GROUP BY 1, 2

ORDER BY frequency DESC;


### bqa5_partition_pruning.sql
**Business Question**: Are there queries performing full table scans on 'historical_travel_time' instead of using the 'record_time' partition filter?
Use Case: Detects 'expensive' behavior. Since RMI datasets are partitioned by day on 'record_time', any query that doesn't include a temporal filter will scan the entire history, significantly increasing costs.
Product Stage: GA (Uses BigQuery INFORMATION_SCHEMA)
Estimated Bytes Processed: N/A (Metadata Query)

In [ ]:
%%bigquery --project {project_id} df_bqa5_partition_pruning
/*
  AUDIT PATTERN: Pruning Heuristics
  This query identifies large scans on the historical table. 
  It calculates if the 'total_bytes_processed' for a job is disproportionately 
  large compared to the typical size of a single daily partition.
  
  Note: Replace 'region-us' with your actual BigQuery region.
*/

SELECT
  job_id,
  user_email,
  query,
  -- Convert bytes to GB for readable performance auditing
  total_bytes_processed / POW(1024, 3) AS gb_processed,
  creation_time
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY'
  AND statement_type = 'SELECT'
  AND query LIKE '%historical_travel_time%'
  -- Heuristic: Trigger audit if scan volume exceeds 100 GB (adjustable baseline)
  -- This suggests the user might have missed a partition pruning filter (record_time)
  AND total_bytes_processed > 100 * POW(1024, 3) 
ORDER BY gb_processed DESC
LIMIT 10;


### bqa6_data_complexity_audit.sql
**Business Question**: What is the average spatial complexity (vertex count) and metadata size (routeAttributes) of my actual records?
Product Stage: GA
Estimated Bytes Processed: ~450 MB (Full scan of geometry and attributes)

In [ ]:
%%bigquery --project {project_id} df_bqa6_data_complexity_audit
/*
  This query performs a deep audit of the data payload. 
  It is useful for understanding the impact of route precision and 
  custom attributes on storage and processing costs.
*/

-- Historical Spatial Complexity
SELECT
  'historical_travel_time' as table_name,
  COUNT(DISTINCT selected_route_id) as unique_routes,
  AVG(BYTE_LENGTH(ST_ASBINARY(route_geometry))) as avg_geom_bytes,
  AVG(ST_LENGTH(route_geometry) / 1000) as avg_route_length_km,
  AVG(ST_NUMPOINTS(route_geometry)) as avg_num_points,
  CAST(NULL AS FLOAT64) as avg_attr_bytes
FROM `boston_oct_2025_sample_data.historical_travel_time`
WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'

UNION ALL

-- Recent Spatial Complexity (Enriched)
SELECT
  'recent_roads_data' as table_name,
  COUNT(DISTINCT selected_route_id) as unique_routes,
  AVG(BYTE_LENGTH(ST_ASBINARY(route_geometry))) as avg_geom_bytes,
  AVG(ST_LENGTH(route_geometry) / 1000) as avg_route_length_km,
  AVG(ST_NUMPOINTS(route_geometry)) as avg_num_points,
  CAST(NULL AS FLOAT64) as avg_attr_bytes
FROM `boston_oct_2025_sample_data.recent_roads_data`
WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'

UNION ALL

-- Status Metadata Complexity
SELECT
  'routes_status' as table_name,
  COUNT(DISTINCT selected_route_id) as unique_routes,
  CAST(NULL AS FLOAT64) as avg_geom_bytes,
  CAST(NULL AS FLOAT64) as avg_route_length_km,
  CAST(NULL AS FLOAT64) as avg_num_points,
  AVG(BYTE_LENGTH(route_attributes)) as avg_attr_bytes
FROM `boston_oct_2025_sample_data.routes_status`;
